# Bank Customer Segment Classification

This notebook trains classification models to predict which **customer segment** a transaction belongs to. The labels (`Target`) come from the K-Means clustering notebook (`clustering_fraud_detection.ipynb`), which grouped customers into three segments:

| Target | Segment |
|---|---|
| 0 | Working-Age Professionals |
| 1 | Young Students |
| 2 | Senior Customers |

With a trained classifier, new customers can be assigned to a segment directly, without re-running the clustering.

**Workflow**
1. Load the labeled dataset
2. Encode categorical features and split into train/test sets
3. Train and evaluate a Decision Tree (baseline) and Logistic Regression
4. Tune a Random Forest with GridSearchCV
5. Compare results and discuss limitations

## 1. Import Libraries
Data handling (`pandas`), models and evaluation tools (`scikit-learn`), and model persistence (`joblib`).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

## 2. Load the Dataset

The input is `data_clustering_inverse.csv`, exported from the clustering notebook. It contains **1,945 transactions** in their original (unscaled) values, plus the cluster label in the `Target` column.

In [2]:
df = pd.read_csv("data_clustering_inverse.csv")

Features are in original units (e.g. age in years, balance in currency) and categorical columns hold text labels.

In [3]:
df.head()

,TransactionAmount,TransactionType,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,BinnedCustomerAge,Target
0,14.09,Debit,ATM,70.0,Doctor,81.0,1.0,5112.21,High,2
1,376.24,Debit,ATM,68.0,Doctor,141.0,1.0,13758.91,High,2
2,126.29,Debit,Online,19.0,Student,56.0,1.0,1122.35,Low,1
3,184.50,Debit,Online,26.0,Student,25.0,1.0,8569.06,Low,1
4,92.15,Debit,ATM,18.0,Student,172.0,1.0,781.68,Low,1


## 3. Data Preparation

### One-hot encoding
Categorical columns (`TransactionType`, `Channel`, `CustomerOccupation`, `BinnedCustomerAge`) are converted to binary columns with `pd.get_dummies`. Unlike label encoding, one-hot encoding doesn't impose an artificial order on the categories. `drop_first=True` removes one redundant column per category. After encoding, the dataset has **13 features**.

In [4]:
categorical_cols = list(df.select_dtypes(include=['object']).columns)

df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)

df_encoded.head()

C:\Users\selyr\AppData\Local\Temp\ipykernel_15912\1488718090.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = list(df.select_dtypes(include=['object']).columns)


,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,Target,TransactionType_Debit,Channel_Branch,Channel_Online,CustomerOccupation_Engineer,CustomerOccupation_Retired,CustomerOccupation_Student,BinnedCustomerAge_Low,BinnedCustomerAge_Medium
0,14.09,70.0,81.0,1.0,5112.21,2,True,False,False,False,False,False,False,False
1,376.24,68.0,141.0,1.0,13758.91,2,True,False,False,False,False,False,False,False
2,126.29,19.0,56.0,1.0,1122.35,1,True,False,True,False,False,True,True,False
3,184.50,26.0,25.0,1.0,8569.06,1,True,False,True,False,False,True,True,False
4,92.15,18.0,172.0,1.0,781.68,1,True,False,False,False,False,True,True,False


### Train/test split
The data is split 80/20 (**1,556 train**, **389 test**). `stratify=y` keeps the same segment proportions in both sets.

In [5]:
X = df_encoded.drop('Target', axis=1)
y = df_encoded['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Jumlah data total: ",len(X))
print("Jumlah data latih: ",len(X_train))
print("Jumlah data test: ",len(X_test))

Jumlah data total:  1945
Jumlah data latih:  1556
Jumlah data test:  389


## 4. Model Training and Evaluation

### Baseline: Decision Tree
A Decision Tree is simple and easy to interpret, which makes it a good baseline to compare other models against.

In [6]:
decision_tree_model = DecisionTreeClassifier(random_state=42)

decision_tree_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

Save the trained Decision Tree.

In [7]:
joblib.dump(decision_tree_model, 'decision_tree_model.h5')

['decision_tree_model.h5']

### Logistic Regression
K-Means assigns each point to its nearest centroid, so the boundaries between clusters are **linear**. A linear model like Logistic Regression should therefore be able to separate the segments well.

Unlike tree-based models, Logistic Regression is sensitive to feature scale (e.g. `AccountBalance` goes up to ~15,000 while `CustomerAge` stays between 18 and 80). The model is wrapped in a `Pipeline` with `StandardScaler`, so the scaler is fit only on the training data and no information leaks from the test set.

In [8]:
new_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))

new_model.fit(X_train, y_train)

c:\Users\selyr\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, random_state=42))])

### Evaluate both models
Precision, recall, and F1-score are reported for each segment on the test set.

In [9]:
y_pred_dt = decision_tree_model.predict(X_test)
y_pred_lr = new_model.predict(X_test)

print("Decision Tree Performance")
print(classification_report(y_test, y_pred_dt))

print("="*50)

print("Logistic Regression Performance")
print(classification_report(y_test, y_pred_lr))

Decision Tree Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       160
           1       1.00      1.00      1.00       102
           2       1.00      1.00      1.00       127

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

Logistic Regression Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       160
           1       1.00      1.00      1.00       102
           2       1.00      1.00      1.00       127

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



Save the Logistic Regression pipeline (scaler + model together).

In [10]:
joblib.dump(new_model, 'logisticregression_classification.h5')

['logisticregression_classification.h5']

## 5. Hyperparameter Tuning: Random Forest

A Random Forest is tuned with `GridSearchCV` (5-fold cross-validation) over 36 combinations of:
- `n_estimators`: number of trees
- `max_depth`: maximum depth of each tree
- `min_samples_split`: minimum samples needed to split a node

The best configuration is a small model (**50 trees, max depth 5**) with a cross-validation accuracy of **0.999**, which shows that the segments are easy to separate.

In [11]:
params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    }

rf_model_tuned = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42),
    param_grid = params,
    cv = 5,
    scoring = 'accuracy'
)

rf_model_tuned.fit(X_train, y_train)
print("Best parameters:", rf_model_tuned.best_params_)
print("Best CV score:", round(rf_model_tuned.best_score_, 3))


Best parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Best CV score: 0.999


### Evaluate the tuned model on the test set

In [12]:
y_pred_tuning = rf_model_tuned.predict(X_test)

print("Tuned Model Performance")
print(classification_report(y_test, y_pred_tuning))

Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       160
           1       1.00      1.00      1.00       102
           2       1.00      1.00      1.00       127

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



Save the tuned Random Forest.

In [13]:
joblib.dump(rf_model_tuned, 'tuning_classification.h5')

['tuning_classification.h5']

## 6. Results and Conclusion

| Model | Accuracy | Macro F1 |
|---|---|---|
| Decision Tree | 1.00 | 1.00 |
| Logistic Regression | 1.00 | 1.00 |
| Random Forest (tuned) | 1.00 | 1.00 |

All three models classify the test set perfectly.

### Why is the accuracy 100%?
This is **not overfitting**; it is expected from how the labels were made. `Target` was produced by K-Means from these same features, so each segment is defined by clear rules (mainly `CustomerAge`, `CustomerOccupation`, and `AccountBalance`). The classifiers only need to learn those rules again. Even a small Random Forest (depth 5) and a linear model get every test sample right.

The models are useful for **assigning new customers to an existing segment quickly**, but the perfect score does not mean the models can detect fraud.

### Limitations and future work
- **The labels come from clustering, not ground truth.** The models can only be as meaningful as the segments. Evaluating them on real outcome labels (for example, confirmed fraud cases) would give a more realistic measure of performance.
- **No fraud signal in the data.** Outlier removal in the clustering notebook left `LoginAttempts = 1` for every row, so the models never see suspicious login behavior.
- **Redundant age features.** `CustomerAge` and `BinnedCustomerAge` carry the same information. Dropping one would simplify the model without losing accuracy.